In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")
from databricks_connector import get_table

target_col = "over_15"
odds_col: str = "custom_odd"


features = [
 'goalNoGoal_chance_goal',
 'goalNoGoal_chance_goalHome',
 'goalNoGoal_chance_goalAway',
 'goalNoGoal_multigoal_m13',
 'goalNoGoal_multigoal_m14',
 'goalNoGoal_multigoal_m24',
 'goalNoGoal_multigoal_m13Home',
 'goalNoGoal_multigoal_m13Away',
 'goalNoGoal_multigoal_m24Home',
 'goalNoGoal_multigoal_m24Away',
 'goalNoGoal_quote_realGG',
 'goalNoGoal_quote_initialGG',
 'goalNoGoal_quote_initialNG',
 'goalNoGoal_quote_currentGG',
 'goalNoGoal_quote_currentNG',
 'goalNoGoal_quote_diffRealCurrGG',
 'goalNoGoal_quote_diffRealCurrNG',
 'goalNoGoal_quote_diffInitialCurrGG',
 'goalNoGoal_quote_diffInitialCurrNG',
 'goalNoGoal_comparison_affini',
 'goalNoGoal_comparison_flashback',
 'goalNoGoal_stats_avgGoalHome',
 'goalNoGoal_stats_avgGoalTakenHome',
 'goalNoGoal_stats_avgGoalAway',
 'goalNoGoal_stats_avgGoalTakenAway',
 'goalNoGoal_flashback_goal',
 'goalNoGoal_flashback_m13',
 'goalNoGoal_flashback_m24',
 'goalNoGoal_flashback_m35',
 'underOver_chance_over05HT',
 'underOver_chance_over052HT',
 'underOver_chance_over15HT',
 'underOver_chance_over15',
 'underOver_chance_over25',
 'underOver_chance_over35',
 'underOver_chance_over45',
 'underOver_quote_realO',
 'underOver_quote_initialU',
 'underOver_quote_initialO',
 'underOver_quote_currentU',
 'underOver_quote_currentO',
 'underOver_quote_diffRealCurrU',
 'underOver_quote_diffRealCurrO',
 'underOver_quote_diffInitialCurrU',
 'underOver_quote_diffInitialCurrO',
 'underOver_comparison_affini',
 'underOver_comparison_flashback',
 'underOver_flashback_under05HT',
 'underOver_flashback_over05HT',
 'underOver_flashback_under15',
 'underOver_flashback_over15',
 'underOver_flashback_under25',
 'underOver_flashback_over25',
 'underOver_flashback_under35',
 'underOver_flashback_over35',
 'evaluation_valScala',
 'evaluation_valMetrica',
 'chance1x2_quote_current1',
 'chance1x2_quote_current2'
 ]

In [3]:
# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()
df[odds_col] = 1.3

In [4]:
def time_train_val_split(
    df,
    time_col="time",
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    sort=True
):
    """
    Split temporale ordinato per train/val/test.

    Parametri
    ----------
    df : pd.DataFrame
    time_col : str
        Nome della colonna temporale.
    train_size, val_size, test_size : float
        Devono sommare a 1.0
    sort : bool
        Se True ordina per time_col crescente.

    Ritorna
    -------
    train_df, val_df, test_df
    """

    if round(train_size + val_size + test_size, 10) != 1.0:
        raise ValueError("train_size + val_size + test_size deve fare 1.0")

    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], utc=True, errors="coerce")

    out = out.dropna(subset=[time_col])

    if sort:
        out = out.sort_values(time_col).reset_index(drop=True)

    n = len(out)
    train_end = int(n * train_size)
    val_end = train_end + int(n * val_size)

    train_df = out.iloc[:train_end].copy()
    val_df = out.iloc[train_end:val_end].copy()
    test_df = out.iloc[val_end:].copy()

    return train_df, val_df, test_df


def rank_univariato_direzione(
    train_df,
    val_df,
    features,
    target_col="y",
    odds_col="quota",
    min_bets_train=80,
    min_bets_val=40,
    percentiles=range(5, 96, 5),
):
    """
    Ranking univariato dei KPI SENZA fissare la soglia finale.
    Per ogni feature:
      - prova soglie candidate solo per capire la direzione migliore
      - assegna uno score robusto
      - ritorna la direzione migliore e una threshold_init opzionale

    Output columns:
      feature, direction, threshold_init,
      n_train, n_val, ev_train, ev_val, score
    """

    train = train_df.copy()
    val = val_df.copy()

    train["return"] = np.where(train[target_col] == 1, train[odds_col] - 1.0, -1.0)
    val["return"] = np.where(val[target_col] == 1, val[odds_col] - 1.0, -1.0)

    rows = []

    for feat in features:
        tr = train[[feat, "return"]].dropna()
        va = val[[feat, "return"]].dropna()

        if len(tr) < min_bets_train * 2 or len(va) < min_bets_val:
            continue

        thresholds = np.unique(np.percentile(tr[feat], list(percentiles)))
        best = None

        for t in thresholds:
            for direction in (">=", "<="):
                if direction == ">=":
                    tr_sub = tr[tr[feat] >= t]
                    va_sub = va[va[feat] >= t]
                else:
                    tr_sub = tr[tr[feat] <= t]
                    va_sub = va[va[feat] <= t]

                n_train = len(tr_sub)
                n_val = len(va_sub)

                if n_train < min_bets_train or n_val < min_bets_val:
                    continue

                ev_train = tr_sub["return"].mean()
                ev_val = va_sub["return"].mean()

                # tieni solo segnali coerenti
                if ev_train <= 0 or ev_val <= 0:
                    continue

                score = min(ev_train, ev_val) * np.sqrt(min(n_train, n_val))

                row = {
                    "feature": feat,
                    "direction": direction,
                    "threshold_init": float(t),
                    "n_train": int(n_train),
                    "n_val": int(n_val),
                    "ev_train": float(ev_train),
                    "ev_val": float(ev_val),
                    "score": float(score),
                }

                if best is None or row["score"] > best["score"]:
                    best = row

        if best is not None:
            rows.append(best)

    if not rows:
        return pd.DataFrame(columns=[
            "feature", "direction", "threshold_init",
            "n_train", "n_val", "ev_train", "ev_val", "score"
        ])

    return (
        pd.DataFrame(rows)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )


def rank_multivariato_greedy(
    train_df,
    val_df,
    top_kpi_df,
    target_col="y",
    odds_col="quota",
    max_features=4,
    min_bets_train=80,
    min_bets_val=40,
    percentiles=range(5, 96, 5),
):
    """
    Forward greedy multivariato.
    
    Input:
      - train_df, val_df
      - top_kpi_df: output di rank_univariato_direzione(...)
        deve avere almeno: feature, direction
    
    Output:
      dict con:
        - selected_rules
        - best_score
        - train_stats
        - val_stats
    """

    train = train_df.copy()
    val = val_df.copy()

    train["return"] = np.where(train[target_col] == 1, train[odds_col] - 1.0, -1.0)
    val["return"] = np.where(val[target_col] == 1, val[odds_col] - 1.0, -1.0)

    candidate_features = top_kpi_df["feature"].tolist()
    direction_map = dict(zip(top_kpi_df["feature"], top_kpi_df["direction"]))

    selected_rules = []
    used_features = set()
    best_score = -np.inf
    best_state = None

    def apply_rules(df, rules):
        mask = pd.Series(True, index=df.index)
        for rule in rules:
            feat = rule["feature"]
            direction = rule["direction"]
            thr = rule["threshold"]

            if direction == ">=":
                mask &= df[feat] >= thr
            else:
                mask &= df[feat] <= thr

        return df[mask].copy()

    def compute_stats(df):
        n = len(df)
        if n == 0:
            return {"n_bets": 0, "ev": np.nan, "profit": 0.0}
        return {
            "n_bets": int(n),
            "ev": float(df["return"].mean()),
            "profit": float(df["return"].sum())
        }

    for _ in range(max_features):
        best_candidate = None
        best_candidate_score = best_score

        for feat in candidate_features:
            if feat in used_features:
                continue

            direction = direction_map[feat]

            tr_current = apply_rules(train, selected_rules)
            va_current = apply_rules(val, selected_rules)

            if feat not in tr_current.columns or feat not in va_current.columns:
                continue

            tr_non_null = tr_current[[feat, "return"]].dropna()
            va_non_null = va_current[[feat, "return"]].dropna()

            if len(tr_non_null) < min_bets_train * 2 or len(va_non_null) < min_bets_val:
                continue

            thresholds = np.unique(np.percentile(tr_non_null[feat], list(percentiles)))

            for t in thresholds:
                candidate_rule = {
                    "feature": feat,
                    "direction": direction,
                    "threshold": float(t)
                }

                trial_rules = selected_rules + [candidate_rule]

                tr_sub = apply_rules(train, trial_rules)
                va_sub = apply_rules(val, trial_rules)

                n_train = len(tr_sub)
                n_val = len(va_sub)

                if n_train < min_bets_train or n_val < min_bets_val:
                    continue

                ev_train = tr_sub["return"].mean()
                ev_val = va_sub["return"].mean()

                # filtro di robustezza
                if ev_train <= 0 or ev_val <= 0:
                    continue

                score = min(ev_train, ev_val) * np.sqrt(min(n_train, n_val))

                if score > best_candidate_score:
                    best_candidate_score = score
                    best_candidate = {
                        "rule": candidate_rule,
                        "score": float(score),
                        "train_stats": compute_stats(tr_sub),
                        "val_stats": compute_stats(va_sub),
                    }

        if best_candidate is None:
            break

        selected_rules.append(best_candidate["rule"])
        used_features.add(best_candidate["rule"]["feature"])
        best_score = best_candidate["score"]

        best_state = {
            "selected_rules": selected_rules.copy(),
            "best_score": best_score,
            "train_stats": best_candidate["train_stats"],
            "val_stats": best_candidate["val_stats"],
        }

    if best_state is None:
        return {
            "selected_rules": [],
            "best_score": None,
            "train_stats": None,
            "val_stats": None,
        }

    return best_state


def applica_regole(df, rules, target_col, odds_col):
    out = df.copy()
    out["return"] = np.where(out[target_col] == 1, out[odds_col] - 1.0, -1.0)

    mask = pd.Series(True, index=out.index)

    for rule in rules:
        if rule["direction"] == ">=":
            mask &= out[rule["feature"]] >= rule["threshold"]
        else:
            mask &= out[rule["feature"]] <= rule["threshold"]

    sel = out[mask].copy()

    stats = {
        "n_bets": int(len(sel)),
        "win_rate": float(sel[target_col].mean()) if len(sel) > 0 else np.nan,
        "ev": float(sel["return"].mean()) if len(sel) > 0 else np.nan,
        "profit": float(sel["return"].sum()) if len(sel) > 0 else 0.0,
        "quota_media": float(sel[odds_col].mean()) if len(sel) > 0 else np.nan,
    }

    return sel, stats

In [1]:
# Main

train_df, val_df, test_df = time_train_val_split(
    df,
    time_col="time",
    train_size=0.6,
    val_size=0.2,
    test_size=0.2
)


rank_df = rank_univariato_direzione(
    train_df=train_df,
    val_df=val_df,
    features=features,
    target_col=target_col,
    odds_col=odds_col
)

top_kpi = rank_df


result = rank_multivariato_greedy(
    train_df=train_df,
    val_df=val_df,
    top_kpi_df=top_kpi.head(15),   # output della univariata
    target_col=target_col,
    odds_col=odds_col,
    max_features=8,
    min_bets_train=80,
    min_bets_val=40
)

print(result["selected_rules"])
print(result["train_stats"])
print(result["val_stats"])
print(result["best_score"])


test_sel, test_stats = applica_regole(
    test_df,
    result["selected_rules"],
    target_col=target_col,
    odds_col=odds_col
)

print(test_stats)

NameError: name 'time_train_val_split' is not defined